-------------------------------------
# **Fraud Detection in Car Insurance Claims**
-------------------------------------

## GTL Data Science and AI

### Universidad Tecnológica - UTEC

Nombres: Florencia Barrios, Florencia Lucas, María Martinote

--------------------
## **Context**
--------------------

**1. The Problem**

Insurance fraud is a global problem that costs billions of dollars every year. For companies, it’s like a 'silent leak' of money. Detecting fraud manually is too slow, too expensive, and has many errors because criminals are getting smarter.

**2. Our Objective**

The goal of our project is to build an AI model using the 'Fraud Oracle' dataset. We want to automatically identify if a car accident claim is 'Real' or 'Fake' by analyzing patterns in the data that the human eye cannot see.

**3. The Importance**

Our model is important for three reasons:

**Money**: It prevents the company from paying millions in f**ake claims.**

**Efficiency**: It allows the company to approve honest claims much faster.

**Fairness**: It helps keep insurance prices low for honest drivers.

**4. Conclusion**

To sum up, we are using Data Science to make insurance faster, cheaper, and safer for everyone.

------------------
## **Objective**
------------------

Car insurance fraud is a big problem because companies lose money on fake claims, AND this hurts their business results. BUT detecting fraud manually is slow and very expensive. THEREFORE, we are using historical data to build a model that predicts if a claim is real or fraudulent.

-----------------------------
## **Key Questions**
-----------------------------

- Which are the features that explain better fraud behaviour?
- What do fraud cases have in common?
- Is younger people more probable to commit fraud?
- Hypothesis: people who have past claims are more probable to commit fraud.
- Is there a relationship between the days between policy purchase and accident and the fraud.
- Hypothesis: expensive cars should have expensive policies.
- Hypothesis: expensive cars owners tend to commit more fraud (is more expensive the deductible amount).

------------------------------------
## **Dataset Description**
------------------------------------




Definition of Features:

- Month: The month in which the insurance claim was made.
- WeekOfMonth: The week of the month in which the insurance claim was made.
- DayOfWeek: The day of the week on which the insurance claim was made.
- Make: The manufacturer of the vehicle involved in the claim.
- AccidentArea: The area where the accident occurred (e.g., urban, rural).
- DayOfWeekClaimed: The day of the week on which the insurance claim was processed.
- MonthClaimed: The month in which the insurance claim was processed.
- WeekOfMonthClaimed: The week of the month in which the insurance claim was processed.
- Sex: The gender of the policyholder.
- MaritalStatus: The material status of the policyholder.
- Age: The age of the policyholder.
- Fault: Indicates whether the policyholder was at fault in the accident.
- PolicyType: The type of insurance policy (e.g., comprehensive, third-party).
- VehicleCategory: The category of the vehicle (e.g., sedan, SUV).
- VehiclePrice: The price of vehicle.
- FraudFound_P: Indicates whether fraud was detected in the insurance claim.
- PolicyNumber: The unique identifier for the insurance policy.
- RepNumber: The unique identifier for the insurance representative handling the claim.
- Deductible: The amount that the policy holder must pay out of pocket before the insurance company pays the remaining costs.
- DriverRating: The rating of the driver, often based on driving history or other factors.
- Days_Policy_Accident: The number of days since the policy was issued until the accident occurred.
- Days_Policy_Claim: The number of days since the policy was issued until the claim was made.
- PastNumberOfClaims: The number of claims previously made by the policyholder.
- AgeOfVehicle: The age of the vehicle involved in the claim.
- AgeOfPolicyHolder: The age of the policyholder.
- PoliceReportFiled: Indicates whether a police report was filed for the accident.
- WitnessPresent: Indicates whether a witness was present at the scene of the accident.
- AgentType: The type of insurance agent handling the policy (e.g., internal, external)
- NumberOfSuppliments: The number of supplementary documents or claims related to the main claim, categorized into ranges.
- AddressChange_Claim: Indicates whether the address of the policyholder was changed at the time of the claim, categorized into ranges.
- NumberOfCars: The number of cars insured under the policy, categorized into ranges.
- Year: The year in which the claim was made or processed.
BasePolicy: The base policy type (e.g., Liability, Collision, All Perils).

##  **Importing the necessary libraries and overview of the dataset**

In [ ]:
# Library to suppress warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
#import optuna

# To build models for prediction
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,BaggingRegressor
from sklearn.tree import plot_tree
from sklearn.tree import export_text
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error as mape


# To encode categorical variables
from sklearn.preprocessing import LabelEncoder

# For tuning the model
from sklearn.model_selection import GridSearchCV

# To check model performance
from sklearn.metrics import make_scorer,mean_squared_error, r2_score, mean_absolute_error

In [ ]:
# Show all the columns
pd.set_option('display.max_columns', None)

In [ ]:
#google drive
from google.colab import drive
drive.mount('/content/drive')

### **Loading the dataset**

In [ ]:
#df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/fraud_oracle.csv')
#df = pd.read_csv('/content/drive/MyDrive/MIT 12-23 enero/fraud_oracle.csv') #flolu
df = pd.read_csv('/content/drive/MyDrive/MIT Enero 2026/Fraud Detection Project/fraud_oracle.csv') #floba

Questions to answer with this data:

- Which are the features that explain better fraud behaviour?
- What do fraud cases have in common?
- Is younger people more probable to commit fraud?
- Hypothesis: people who have past claims are more probable to commit fraud.
- Is there a relationship between the days between policy purchase and accident and the fraud.
- Hypothesis: expensive cars should have expensive policies.
- Hypothesis: expensive cars owners tend to commit more fraud (is more expensive the deductible amount.

Which variables to include/exclude:
Initially we would analyze all the variables. EDA to see if there are not relevant (eg. all the same value)

Outliers:
check with histograms in the univariate analysis.

Errors/nulls:
check

## **Overview**

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df['Year'].value_counts()

In [ ]:
df.describe().T

Observations:

- Age 0 is an outlier (error?)
- Policy number is an identifier, so it adds no value to the analysis. Just check for duplicated.
- Rep number (identifier of the representative).
- Years between 1994 and 1996.

In [ ]:
print(df.columns)

In [ ]:
df.info()

Observations:

- There are no missing values in the dataset.
- FraudFound_P is the target,

**By default, the describe() function shows the summary of numeric variables only. Let's check the summary of non-numeric variables.**  

In [ ]:
df.describe(exclude = 'number').T

Observation
- 13.000 are men in the dataset


::**Let's check the count of each unique category in each of the categorical variables.**

In [ ]:
# Making a list of all object variables
cat_col = [col for col in df.columns if df[col].dtype == 'object']

# Printing number of count of each unique value in each column
for column in cat_col:
    print(df[column].value_counts())

    print('-' * 50)

## **Duplicate check**

In [ ]:
#duplicated variables
df.duplicated().sum()

In [ ]:
# check duplicated of ID
df['PolicyNumber'].duplicated().sum()

There are no duplicated values. We will drop policy number because it is an identification number.

In [ ]:
# drop policy number
df.drop('PolicyNumber', axis=1, inplace=True)

## **Feature Engineering**

In [ ]:
df.head()

**Accident and Claim Date**

In [ ]:
import pandas as pd

# 1. Mapeo de meses a números
month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}

def create_dates(df):
    # Convertir nombres de meses a números
    df['Month_num'] = df['Month'].map(month_map)
    df['MonthClaimed_num'] = df['MonthClaimed'].map(month_map)

    # Manejar posibles errores en MonthClaimed (a veces viene como 0)
    df['MonthClaimed_num'] = df['MonthClaimed_num'].fillna(1).astype(int)

    # Crear día aproximado basado en la semana del mes
    # (Semana 1 -> día 1, Semana 2 -> día 8, etc.)
    df['Day_Accident'] = ((df['WeekOfMonth'] - 1) * 7 + 1).clip(upper=28)
    df['Day_Claim'] = ((df['WeekOfMonthClaimed'] - 1) * 7 + 1).clip(upper=28)

    # Construir AccidentDate
    df['AccidentDate'] = pd.to_datetime(df[['Year', 'Month_num', 'Day_Accident']].rename(
        columns={'Year': 'year', 'Month_num': 'month', 'Day_Accident': 'day'}))

    # Lógica para el año del Claim:
    # Si el mes del reclamo es menor al mes del accidente, asumimos que fue el año siguiente
    df['YearClaimed'] = df.apply(
        lambda x: x['Year'] + 1 if x['MonthClaimed_num'] < x['Month_num'] else x['Year'], axis=1
    )

    # Construir ClaimDate
    df['ClaimDate'] = pd.to_datetime(df[['YearClaimed', 'MonthClaimed_num', 'Day_Claim']].rename(
        columns={'YearClaimed': 'year', 'MonthClaimed_num': 'month', 'Day_Claim': 'day'}))

    return df

df = create_dates(df)

In [ ]:
df.head()

In [ ]:
# Calcular la diferencia en días
df['Days_to_Report'] = (df['ClaimDate'] - df['AccidentDate']).dt.days

# Validación: Si por la estimación de semanas algún valor da negativo, lo llevamos a 0
df['Days_to_Report'] = df['Days_to_Report'].apply(lambda x: x if x >= 0 else 0)

# Ver los primeros resultados
print(df[['AccidentDate', 'ClaimDate', 'Days_to_Report', 'FraudFound_P']].head())

In [ ]:
df['Days_to_Report'].describe()

Most of the cases are reported on the same day of the accident.

**Dummy difference between driver´s age and policy holder age**

In [ ]:
def create_driver_match_dummy(df):
    # 1. Función para convertir edad numérica al rango de AgeOfPolicyHolder
    def map_age_to_range(age):
        if age == 0: return "Unknown" # Manejamos el 0 como dato faltante
        elif 16 <= age <= 17: return "16 to 17"
        elif 18 <= age <= 20: return "18 to 20"
        elif 21 <= age <= 25: return "21 to 25"
        elif 26 <= age <= 30: return "26 to 30"
        elif 31 <= age <= 35: return "31 to 35"
        elif 36 <= age <= 40: return "36 to 40"
        elif 41 <= age <= 50: return "41 to 50"
        elif 51 <= age <= 65: return "51 to 65"
        elif age > 65: return "over 65"
        else: return "Unknown"

    # 2. Aplicamos el mapeo a la edad del conductor (Age)
    df['Age_Range_Driver'] = df['Age'].apply(map_age_to_range)

    # 3. Creamos la variable Dummy: 1 si coinciden, 0 si no coinciden
    # Nota: Si la edad es "Unknown" (0), marcamos como 0 (no coincide o no sabemos)
    df['Is_PolicyHolder_Driving'] = np.where(
        (df['Age_Range_Driver'] == df['AgeOfPolicyHolder']) & (df['Age_Range_Driver'] != "Unknown"),
        1, 0
    )

    return df

df = create_driver_match_dummy(df)

In [ ]:
df.head()

In [ ]:
df['Is_PolicyHolder_Driving'].value_counts()

In [ ]:
# cross table policy holder driving and fraud with percentage
pd.crosstab(df['Is_PolicyHolder_Driving'], df['FraudFound_P'], normalize='index')

**Convert categorical variables into numerical**

In [ ]:
df.info()

In [ ]:
# one hot encoding for Make

## **Exploratory Data Analysis: Univariate**

In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

def histogram_boxplot(feature, figsize=(15, 10), bins="auto"):
    """ Boxplot and histogram combined
    feature: 1-d feature array
    figsize: size of fig (default (15, 10))
    bins: number of bins (default "auto")
    """
    f, (ax_box, ax_hist) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid
        sharex=True,  # The X-axis will be shared among all the subplots
        gridspec_kw={"height_ratios": (.25, .75)},
        figsize=figsize
    )

    # Creating the subplots
    # Boxplot will be created and the mean value of the column will be indicated using some symbol
    sns.boxplot(x=feature, ax=ax_box, showmeans=True, color='red')

    # For histogram
    sns.histplot(x=feature, kde=False, ax=ax_hist, bins=bins)
    ax_hist.axvline(np.mean(feature), color='g', linestyle='--')      # Add mean to the histogram
    ax_hist.axvline(np.median(feature), color='black', linestyle='-') # Add median to the histogram

    plt.show()

In [ ]:
def bar_perc(data, z):
    total = len(data[z]) # Length of the column
    plt.figure(figsize = (15, 5))

    # Convert the column to a categorical data type
    data[z] = data[z].astype('category')

    ax = sns.countplot(x=z, data=data, palette='Paired', order=data[z].value_counts().index)

    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height() / total) # Percentage of each class
        x = p.get_x() + p.get_width() / 2 - 0.05                    # Width of the plot
        y = p.get_y() + p.get_height()                              # Height of the plot
        ax.annotate(percentage, (x, y), size = 12)                  # Annotate the percentage

    plt.show()                                                      # Display the plot

#### Fraud

In [ ]:
import matplotlib.pyplot as plt

# 1. Contamos los valores
fraud_counts = df['FraudFound_P'].value_counts()

# 2. Definimos etiquetas (0 es Legit, 1 es Fraud)
labels = ['Legit' if x == 0 else 'Fraud' for x in fraud_counts.index]

# 3. Definimos colores pastel específicos:
# #b2f2bb es un verde pastel (Caso Feliz)
# #ffadad es un rojo/rosado pastel (Fraude)
# Usamos una lista de comprensión para asegurar que el color siga al índice
colors = ['#b2f2bb' if x == 0 else '#ffadad' for x in fraud_counts.index]

# 4. Creamos el gráfico
plt.figure(figsize=(5, 5))
plt.pie(fraud_counts,
        labels=labels,
        autopct='%1.1f%%',
        startangle=140,
        colors=colors,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2}) # Borde blanco para mayor limpieza

plt.title('Proportion of Fraudulent vs. Legitimate Claims', fontsize=14, fontweight='bold')
plt.show()

Only 6% of claims are detected as Fraud. The dataset is unbalanced.

#### Days to report

In [ ]:
histogram_boxplot(df.Days_to_Report)

**Observations:**
* The distribution of Days to Report is highly right-skewed.
* Most of the claims are made in the moment or a few days after the accident.
* There are a lot of outliers in this variable.

#### Age of the driver

In [ ]:
histogram_boxplot(df.Age)

- The age is normally distributed
- There are ages=0. We should analyze those cases with more detail

In [ ]:
# Apply bar_perc to all object variables
for col in cat_col:
    bar_perc(df, col)

In [ ]:
bar_perc(df, 'Fault')

In [ ]:
bar_perc(df, 'AgeOfPolicyHolder')

In [ ]:
bar_perc(df, 'Sex')

## **Exploratory Data Analysis: Multivariate**

## **Relationship between frauds and time based variables**

### **Frauds across Months**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Definimos el orden correcto de los meses
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
df['Month'] = pd.Categorical(df['Month'], categories=month_order, ordered=True)

# 2. Configuramos el gráfico
plt.figure(figsize=(15, 7))

# Graficamos la tasa de fraude (en azul para que contraste con la línea roja)
sns.lineplot(x="Month", y="FraudFound_P", data=df,
             estimator=lambda x: np.mean(x) * 100,
             color="#1f77b4", # Azul profesional
             linewidth=3,
             marker='o',
             markersize=8,
             errorbar=None)

# 3. Agregamos la línea de referencia en el 6% (Promedio General)
plt.axhline(y=6, color='red', linestyle='--', linewidth=2, label='Average Fraud Rate (6%)')

# 4. Ajustamos el eje Y para notar la variación (Zoom entre 3% y 8%)
plt.ylim(3, 8)

# 5. Estética final
plt.title('Fraud Rate Seasonality vs. Average', fontsize=16, pad=20)
plt.ylabel('Fraud Rate (%)', fontsize=12)
plt.xlabel('Month', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, axis='y', linestyle=':', alpha=0.6)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# 1. Definimos el orden cronológico de los días
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['DayOfWeek'] = pd.Categorical(df['DayOfWeek'], categories=day_order, ordered=True)

# 2. Configuramos el gráfico
plt.figure(figsize=(15, 7))

# Graficamos la tasa de fraude (Azul profesional)
sns.lineplot(x="DayOfWeek", y="FraudFound_P", data=df,
             estimator=lambda x: np.mean(x) * 100,
             color="#1f77b4",
             linewidth=3,
             marker='o',
             markersize=8,
             errorbar=None)

# 3. Línea de referencia en el 6% (Promedio General)
plt.axhline(y=6, color='red', linestyle='--', linewidth=2, label='Average Fraud Rate (6%)')

# 4. Zoom en el eje Y (3% al 8%) para ver la sensibilidad
plt.ylim(3, 8)

# 5. Estética final
plt.title('Fraud Rate by Day of the Week vs. Average', fontsize=16, pad=20)
plt.ylabel('Fraud Rate (%)', fontsize=12)
plt.xlabel('Day of the Accident', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, axis='y', linestyle=':', alpha=0.6)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. Aseguramos el orden de las categorías (por si no se guardó antes)
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

df['Month'] = pd.Categorical(df['Month'], categories=month_order, ordered=True)
df['DayOfWeek'] = pd.Categorical(df['DayOfWeek'], categories=day_order, ordered=True)

# 2. Creamos una tabla pivote con el promedio de fraude (tasa)
# Multiplicamos por 100 para tener porcentajes
heatmap_data = df.pivot_table(index='Month',
                             columns='DayOfWeek',
                             values='FraudFound_P',
                             aggfunc='mean') * 100

# 3. Dibujamos el Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_data,
            annot=True,      # Muestra el número dentro del cuadro
            fmt=".1f",       # Un decimal
            cmap='YlOrRd',   # Escala de Amarillo a Rojo (Rojo = Más Fraude)
            linewidths=.5)

plt.title('Fraud Risk Heatmap: Month vs. Day of Week (%)', fontsize=16, pad=20)
plt.ylabel('Month of Accident', fontsize=12)
plt.xlabel('Day of Week', fontsize=12)
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Seleccionamos las variables clave (incluyendo las nuevas)
# Nota: Asegúrense de haber corrido los códigos anteriores para crear estas columnas
features_to_check = [
    'FraudFound_P',
    'Days_to_Report',
    'Is_PolicyHolder_Driving',
    'Deductible',
    'DriverRating',
    'Year'
]

# Calculamos la matriz de correlación
corr_matrix = df[features_to_check].corr()

# 2. Configuramos el gráfico
plt.figure(figsize=(12, 10))

# Creamos una máscara para ver solo la mitad inferior (es más limpio)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix,
            mask=mask,
            annot=True,
            fmt=".2f",
            cmap='RdBu_r', # Rojo para pos, Azul para neg
            center=0,
            linewidths=.5)

plt.title('Correlation Matrix: Key Fraud Predictors', fontsize=16, pad=20)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Configuramos el estilo
plt.figure(figsize=(12, 7))

# 2. Creamos el Boxplot
# Usamos showfliers=False para que los valores extremos (outliers) no aplasten el gráfico
# y podamos ver bien dónde está la mayoría de la gente.
sns.boxplot(x='FraudFound_P', y='Days_to_Report', data=df,
            palette=['#b2f2bb', '#ffadad'],
            showfliers=False,
            linewidth=2)

# 3. Personalizamos etiquetas
plt.xticks([0, 1], ['Legit Claims', 'Fraudulent Claims'], fontsize=12)
plt.title('Distribution of Reporting Lag: Legit vs. Fraud', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Days from Accident to Claim', fontsize=12)
plt.xlabel('Claim Type', fontsize=12)

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Agrupamos y calculamos Volumen y Tasa
summary = df.groupby('Make')['FraudFound_P'].agg(['count', 'mean']).reset_index()
summary['Fraud_Rate_Pct'] = summary['mean'] * 100

# 2. ORDENAMOS POR TASA DE FRAUDE (de mayor a menor)
summary = summary.sort_values('Fraud_Rate_Pct', ascending=False)

# 3. Creamos el gráfico de doble eje
fig, ax1 = plt.subplots(figsize=(16, 8))
ax2 = ax1.twinx()

# Barras: Volumen (en el eje izquierdo)
# El parámetro 'order' es clave para que las barras sigan el orden de la tasa
sns.barplot(x='Make', y='count', data=summary, ax=ax1,
            color='#d1d1d1', alpha=0.6, order=summary['Make'])

# Línea: Tasa de Fraude (en el eje derecho)
sns.lineplot(x='Make', y='Fraud_Rate_Pct', data=summary, ax=ax2,
             color='#d62728', marker='o', markersize=10, linewidth=3, label='Fraud Rate %')

# Línea de referencia del 6%
ax2.axhline(y=6, color='black', linestyle='--', linewidth=2, label='Avg. Rate (6%)')

# 4. Estética y etiquetas
ax1.set_ylabel('Total Claims Volume (Bars)', fontsize=12, color='gray')
ax2.set_ylabel('Fraud Rate % (Line)', fontsize=12, color='#d62728')
ax1.set_xticklabels(summary['Make'], rotation=45)
plt.title('Analysis Sorted by Risk: Fraud Rate vs. Volume', fontsize=16, pad=20)

# Unir leyendas
lines, labels = ax2.get_legend_handles_labels()
ax2.legend(lines, labels, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Definimos el orden lógico del precio
price_order = ['less than 20000', '20000 to 29000', '30000 to 39000',
               '40000 to 59000', '60000 to 69000', 'more than 69000']

# 2. Preparamos los datos
price_summary = df.groupby('VehiclePrice')['FraudFound_P'].mean().reset_index()
price_summary['Fraud_Rate'] = price_summary['FraudFound_P'] * 100

# 3. Configuramos el gráfico
plt.figure(figsize=(12, 8))

# Paleta (rojo en extremos, gris en el medio)
palette = ['#ff4d4d' if (p == 'less than 20000' or p == 'more than 69000') else '#d1d1d1'
           for p in price_order]

# Gráfico de barras
bars = sns.barplot(x='VehiclePrice', y='Fraud_Rate', data=price_summary,
                   order=price_order, palette=palette)

# --- ANOTACIONES CORREGIDAS (Posición óptima) ---

# Flecha en el extremo bajo (Bien ubicada)
plt.annotate('Fraud by Disposal:\nGetting rid of cheap cars', xy=(0, 9.5), xytext=(0.5, 11.5),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1),
             fontsize=10, fontweight='bold', color='#d62728', ha='center')

# Flecha en el extremo alto (CORREGIDA: mucho más cerca de la barra)
# xy=(5, 12) apunta a la punta de la barra.
# xytext=(4.8, 13.5) pone el texto justo encima y un poco a la izquierda para que quede centrado sobre la barra.
plt.annotate('Profit-Driven Fraud:\nHigh payout incentive', xy=(5, 12), xytext=(4.8, 13.5),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1),
             fontsize=10, fontweight='bold', color='#d62728', ha='center')

# Estética final
plt.axhline(y=6, color='black', linestyle='--', alpha=0.5, label='General Average (6%)')
plt.title('The "U-Curve" of Insurance Fraud: Risk at the Extremes', fontsize=16, fontweight='bold', pad=30)
plt.ylabel('Fraud Rate (%)', fontsize=12)
plt.xlabel('Vehicle Price Category', fontsize=12)
# Damos un poco más de aire arriba para que el texto respire
plt.ylim(0, max(price_summary['Fraud_Rate']) + 5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

def identify_low_variance_features(df, threshold=0.90):
    low_variance_list = []

    for col in df.columns:
        # Calculamos la frecuencia del valor más común
        most_common_value_pct = df[col].value_counts(normalize=True).iloc[0]

        if most_common_value_pct > threshold:
            # Guardamos el nombre, el valor dominante y su porcentaje
            dominant_value = df[col].value_counts().index[0]
            low_variance_list.append({
                'Variable': col,
                'Dominant Value': dominant_value,
                'Percentage': round(most_common_value_pct * 100, 2)
            })

    # Convertimos a DataFrame para verlo ordenado
    low_var_df = pd.DataFrame(low_variance_list).sort_values(by='Percentage', ascending=False)
    return low_var_df

# Ejecutamos la función
variables_a_revisar = identify_low_variance_features(df, threshold=0.90)

# Mostramos el resultado
print(f"Se encontraron {len(variables_a_revisar)} variables con un valor dominante mayor al 90%:")
print(variables_a_revisar)

In [ ]:
import pandas as pd

def find_rare_categories(df, min_cases=10):
    rare_categories_list = []

    # Seleccionamos solo las columnas categóricas (objetos o categorías)
    cat_cols = df.select_dtypes(include=['object', 'category']).columns

    for col in cat_cols:
        # Obtenemos el conteo de cada categoría
        counts = df[col].value_counts()

        # Filtramos las que tienen menos de los casos definidos
        rare_ones = counts[counts < min_cases]

        for category, count in rare_ones.items():
            rare_categories_list.append({
                'Variable': col,
                'Category': category,
                'Count': count
            })

    # Convertimos a DataFrame para una vista limpia
    rare_df = pd.DataFrame(rare_categories_list)

    if not rare_df.empty:
        return rare_df.sort_values(by=['Variable', 'Count'])
    else:
        return "No se encontraron categorías con menos de " + str(min_cases) + " casos."

# Ejecutamos la función
categorias_raras = find_rare_categories(df, min_cases=10)

# Mostramos el resultado
print(categorias_raras)

In [ ]:
# Cambia 'AgentType' por cualquier variable de la lista que te salió arriba
variable_sospechosa = 'AgentType'

cross_tab = pd.crosstab(df[variable_sospechosa], df['FraudFound_P'], normalize='index') * 100
print(f"--- Análisis de {variable_sospechosa} vs Fraude ---")
print(cross_tab)

In [ ]:
# Cambia 'AgentType' por cualquier variable de la lista que te salió arriba
variable_sospechosa = 'NumberOfCars'

cross_tab = pd.crosstab(df[variable_sospechosa], df['FraudFound_P'], normalize='index') * 100
print(f"--- Análisis de {variable_sospechosa} vs Fraude ---")
print(cross_tab)

In [ ]:
# Cambia 'AgentType' por cualquier variable de la lista que te salió arriba
variable_sospechosa = 'Days_Policy_Claim'

cross_tab = pd.crosstab(df[variable_sospechosa], df['FraudFound_P'], normalize='index') * 100
print(f"--- Análisis de {variable_sospechosa} vs Fraude ---")
print(cross_tab)

In [ ]:
df['Days_Policy_Claim'].value_counts()

Days between the policy purchase and the claim seem to be an important feature to predict fraud. The less days the higher the fraud rate.
There is only 1 case with none, we may drop it.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Definimos el orden lógico de los rangos de tiempo
# (Asegúrate de que estos nombres coincidan exactamente con los de tu df)
policy_claim_order = ['none', '1 to 7', '8 to 15', '15 to 30', 'more than 30']

# 2. Agrupamos y calculamos Volumen y Tasa
summary = df.groupby('Days_Policy_Claim')['FraudFound_P'].agg(['count', 'mean']).reset_index()
summary['Fraud_Rate_Pct'] = summary['mean'] * 100

# Convertimos a categoría para que el eje X respete el orden de tiempo
summary['Days_Policy_Claim'] = pd.Categorical(
    summary['Days_Policy_Claim'],
    categories=policy_claim_order,
    ordered=True
)
summary = summary.sort_values('Days_Policy_Claim')

# 3. Configuramos el gráfico de doble eje
fig, ax1 = plt.subplots(figsize=(12, 7))
ax2 = ax1.twinx()

# --- EJE IZQUIERDO: Volumen de Casos (Barras) ---
sns.barplot(x='Days_Policy_Claim', y='count', data=summary,
            ax=ax1, color='#d1d1d1', alpha=0.6)
ax1.set_ylabel('Total Claims Volume (Bars)', fontsize=12, color='gray')
ax1.tick_params(axis='y', labelcolor='gray')

# --- EJE DERECHO: Tasa de Fraude (Línea) ---
sns.lineplot(x='Days_Policy_Claim', y='Fraud_Rate_Pct', data=summary,
             ax=ax2, color='#d62728', marker='o', markersize=12, linewidth=4, label='Fraud Rate %')

# Línea de referencia del promedio general (6%)
ax2.axhline(y=6, color='black', linestyle='--', linewidth=2, label='Avg. Rate (6%)')

# 4. Estética y Storytelling
plt.title('Fraud Risk by Policy Aging: Days Between Policy and Claim', fontsize=16, fontweight='bold', pad=25)
ax1.set_xlabel('Days from Policy Issue to Accident', fontsize=12)
ax2.set_ylabel('Fraud Rate % (Line)', fontsize=12, color='#d62728')
ax2.set_ylim(0, max(summary['Fraud_Rate_Pct']) + 10) # Damos espacio para ver los picos

# Unir leyendas
lines, labels = ax2.get_legend_handles_labels()
ax2.legend(lines, labels, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Orden lógico de la antigüedad del cambio
addr_order = ['no change', 'under 6 months', '1 to 2 years', '2 to 3 years', '4 to 8 years']

plt.figure(figsize=(12, 6))
addr_summary = df.groupby('AddressChange_Claim')['FraudFound_P'].mean() * 100

# Resaltamos en rojo los cambios recientes (menos de 6 meses)
colors = ['#ff4d4d' if x == 'under 6 months' else '#d1d1d1' for x in addr_order]

sns.barplot(x=addr_summary.index, y=addr_summary.values, order=addr_order, palette=colors)
plt.axhline(y=6, color='black', linestyle='--')
plt.title('Fraud Risk vs. Address Change Status', fontsize=15, fontweight='bold')
plt.ylabel('Fraud Rate (%)')
plt.show()

In [ ]:
# cross table de fraude y de adress change
pd.crosstab(df['AddressChange_Claim'], df['FraudFound_P'])

In [ ]:
plt.figure(figsize=(12, 6))
# Filtramos Age > 0 para evitar el ruido de los datos faltantes que detectaron
sns.kdeplot(data=df[df['Age'] > 0], x='Age', hue='FraudFound_P',
            fill=True, common_norm=False, palette=['#b2f2bb', '#ff4d4d'])

plt.title('Age Distribution: Honest vs. Fraudulent Drivers', fontsize=15, fontweight='bold')
plt.xlabel('Driver Age')
plt.ylabel('Density')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Agrupamos por WitnessPresent y calculamos Volumen y Tasa
witness_summary = df.groupby('WitnessPresent')['FraudFound_P'].agg(['count', 'mean']).reset_index()
witness_summary['Fraud_Rate_Pct'] = witness_summary['mean'] * 100

# 2. Configuramos el gráfico de doble eje
fig, ax1 = plt.subplots(figsize=(10, 7))
ax2 = ax1.twinx()

# --- EJE IZQUIERDO: Barras de Volumen (Aquí se verá ese 90%) ---
sns.barplot(x='WitnessPresent', y='count', data=witness_summary, ax=ax1,
            palette=['#d1d1d1', '#d1d1d1'], alpha=0.6)
ax1.set_ylabel('Total Claims Volume (Bars)', fontsize=12, color='gray')
ax1.tick_params(axis='y', labelcolor='gray')

# --- EJE DERECHO: Línea de Tasa de Fraude ---
sns.lineplot(x='WitnessPresent', y='Fraud_Rate_Pct', data=witness_summary, ax=ax2,
             color='#d62728', marker='o', markersize=12, linewidth=4, label='Fraud Rate %')

# Línea de referencia del 6%
ax2.axhline(y=6, color='black', linestyle='--', linewidth=2, label='Avg. Rate (6%)')

# 3. Estética y Storytelling
plt.title('Witness Presence: Volume vs. Fraud Risk', fontsize=16, fontweight='bold', pad=25)
ax1.set_xlabel('Was a Witness Present?', fontsize=12)
ax2.set_ylabel('Fraud Rate % (Line)', fontsize=12, color='#d62728')
ax2.set_ylim(0, max(witness_summary['Fraud_Rate_Pct']) + 5) # Ajuste de escala

# Unir leyendas
lines, labels = ax2.get_legend_handles_labels()
ax2.legend(lines, labels, loc='upper center')

plt.tight_layout()
plt.show()

In [ ]:
df.columns

In [ ]:
# cross table with policy holder age and fraud
pd.crosstab(df['AgeOfPolicyHolder'], df['FraudFound_P'], normalize='index')

In [ ]:
# 2. Configuramos el gráfico
plt.figure(figsize=(15, 7))

# Graficamos la tasa de fraude (Azul profesional)
sns.lineplot(x="AgeOfPolicyHolder", y="FraudFound_P", data=df,
             estimator=lambda x: np.mean(x) * 100,
             color="#1f77b4",
             linewidth=3,
             marker='o',
             markersize=8,
             errorbar=None)

# 3. Línea de referencia en el 6% (Promedio General)
plt.axhline(y=6, color='red', linestyle='--', linewidth=2, label='Average Fraud Rate (6%)')

# 4. Zoom en el eje Y (3% al 8%) para ver la sensibilidad
plt.ylim(3, 8)

# 5. Estética final
plt.title('Fraud Rate by Day of the Week vs. Average', fontsize=16, pad=20)
plt.ylabel('Fraud Rate (%)', fontsize=12)
plt.xlabel('Day of the Accident', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, axis='y', linestyle=':', alpha=0.6)

plt.show()

## Categorical features conversion to numerical

**Ordinal features**

In [ ]:
import pandas as pd

# Creamos la copia final
df_numeric = df.copy()

# 1. ELIMINAR COLUMNAS NO PREDICTIVAS (IDs)
# Estas no aportan al modelo y pueden causar ruido
cols_to_drop = ['PolicyNumber', 'RepNumber']
df_numeric = df_numeric.drop(columns=[c for c in cols_to_drop if c in df_numeric.columns])

# 2. MAPEOS ORDINALES (Todos los discutidos)
ordinal_mappings = {
    'VehiclePrice': {
        'less than 20000': 1, '20000 to 29000': 2, '30000 to 39000': 3,
        '40000 to 59000': 4, '60000 to 69000': 5, 'more than 69000': 6
    },
    'AgeOfVehicle': {
        'new': 0, '2 years': 1, '3 years': 2, '4 years': 3,
        '5 years': 4, '6 years': 5, '7 years': 6, 'more than 7': 7
    },
    'BasePolicy': {
        'Liability': 1, 'Collision': 2, 'All Perils': 3
    },
    'Month': {'Jan':1, 'Feb':2, 'Mar':3, 'Apr':4, 'May':5, 'Jun':6, 'Jul':7, 'Aug':8, 'Sep':9, 'Oct':10, 'Nov':11, 'Dec':12},
    'MonthClaimed': {'Jan':1, 'Feb':2, 'Mar':3, 'Apr':4, 'May':5, 'Jun':6, 'Jul':7, 'Aug':8, 'Sep':9, 'Oct':10, 'Nov':11, 'Dec':12},
    'DayOfWeek': {'Monday':1, 'Tuesday':2, 'Wednesday':3, 'Thursday':4, 'Friday':5, 'Saturday':6, 'Sunday':7},
    'DayOfWeekClaimed': {'Monday':1, 'Tuesday':2, 'Wednesday':3, 'Thursday':4, 'Friday':5, 'Saturday':6, 'Sunday':7},
    'AgeOfPolicyHolder': {'16 to 17':1, '18 to 20':2, '21 to 25':3, '26 to 30':4, '31 to 35':5, '36 to 40':6, '41 to 50':7, '51 to 65':8, 'over 65':9},
    'PastNumberOfClaims': {'none':0, '1':1, '2 to 4':2, 'more than 4':3},
    'NumberOfSuppliments': {'none':0, '1 to 2':1, '3 to 5':2, 'more than 5':3},
    'NumberOfCars': {'1 vehicle':1, '2 vehicles':2, '3 to 4':3, '5 to 8':4, 'more than 8':5},
    'Days_Policy_Accident': {'none':0, '1 to 7':1, '8 to 15':2, '15 to 30':3, 'more than 30':4},
    'Days_Policy_Claim': {'none':0, '1 to 7':1, '8 to 15':2, '15 to 30':3, 'more than 30':4}
}

for col, mapping in ordinal_mappings.items():
    if col in df_numeric.columns:
        df_numeric[col] = df_numeric[col].map(mapping)
        # Explicitly convert to float to handle potential NaNs without creating a Categorical type issue
        df_numeric[col] = df_numeric[col].astype(float)


# 3. VARIABLES BINARIAS (Tus especificaciones)
binary_mappings = {
    'AccidentArea': {'Urban': 1, 'Rural': 0},
    'Sex': {'Female': 0, 'Male': 1},
    'Fault': {'Policy Holder': 1, 'Third Party': 0},
    'PoliceReportFiled': {'No': 0, 'Yes': 1},
    'WitnessPresent': {'No': 0, 'Yes': 1},
    'AgentType': {'External': 1, 'Internal': 0}
}

for col, mapping in binary_mappings.items():
    if col in df_numeric.columns:
        df_numeric[col] = df_numeric[col].map(mapping)
        # Explicitly convert to float to handle potential NaNs
        df_numeric[col] = df_numeric[col].astype(float)


# 4. ONE-HOT ENCODING (Resto de nominales)
# Variables como 'Make', 'MaritalStatus', etc.
processed_cols = list(ordinal_mappings.keys()) + list(binary_mappings.keys())

# Filter for object or category columns that haven't been processed yet
# Also ensure datetime columns are not included as they are not meant for one-hot encoding
remaining_cats = [col for col in df_numeric.select_dtypes(include=['object', 'category']).columns
                  if col not in processed_cols and df_numeric[col].dtype not in ['datetime64[ns]', 'datetime64[s]']]

df_numeric = pd.get_dummies(df_numeric, columns=remaining_cats, drop_first=True)

# 5. CONTROL DE CALIDAD FINAL
# Llenamos posibles NaNs. Now that all relevant columns are numerical (float, int, or dummy 0/1),
# fillna(0) will correctly replace NaNs with 0 without TypeErrors.
df_numeric = df_numeric.fillna(0)

print(f"Dataset transformado exitosamente. Columnas totales: {df_numeric.shape[1]}")

In [ ]:
# Convertimos todo el DataFrame a tipo entero
df_numeric = df_numeric.astype(int)

# Verificamos que ahora todas las columnas sean 'int64' o 'int32'
print("--- VERIFICACIÓN DE TIPOS DE DATOS ---")
print(df_numeric.dtypes.value_counts())

# Echamos un vistazo a las primeras filas para confirmar los 0 y 1
df_numeric.head()

In [ ]:
# drop auxiliar features
df_numeric=df_numeric.drop(columns=['Year', 'Month_num',	'MonthClaimed_num',	'Day_Accident',	'Day_Claim',	'AccidentDate',	'YearClaimed',
                         'ClaimDate'])

In [ ]:
# correlation matrix
corr_matrix = df_numeric.corr()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Calculamos la matriz de correlación completa
corr_matrix = df_numeric.corr()

# 2. Filtramos: Solo queremos variables que tengan > 0.1 de correlación con FraudFound_P
# Usamos el valor absoluto para incluir tanto correlaciones positivas como negativas
target_corr = corr_matrix['FraudFound_P'].abs()
significant_features = target_corr[target_corr > 0.1].index

# Creamos una sub-matriz solo con esas variables
filtered_corr = df_numeric[significant_features].corr()

# 3. Configuramos el gráfico
plt.figure(figsize=(12, 10))

# Máscara para la mitad superior
mask = np.triu(np.ones_like(filtered_corr, dtype=bool))

# Dibujamos el Heatmap
sns.heatmap(filtered_corr,
            mask=mask,
            annot=True,
            fmt=".2f",
            cmap='RdBu_r',
            center=0,
            linewidths=.5,
            cbar_kws={"shrink": .8})

plt.title('Significant Predictors: Correlations > 0.1', fontsize=16, pad=20)
plt.show()